<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-04-rag/lesson-4.2-diy-rag/notebooks/GCP_Capstone_4.2_DIY_RAG.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.2 DIY RAG Pipeline — Retrieve → Augment → Generate
**Netsetos GenAI Engineering — GCP Capstone**

Complete RAG: embed query, retrieve vectors, augment prompt, generate with citations, track cost.


## Setup
In the DocuMind build, `chunks` is DocuMind's ONE collection — the same one 2.3 seeds, 4.5/4.6 read and the 12.5 ingest worker writes in production — with one document shape: `tenant_id`, `text`, `source_uri`, `page_start`, `doc_type`, `embedding`.

The corpus it holds is **DocuMind's documents, not demo sentences**: `deploy/evals/corpus/` in the course repo. ACME carries its (synthetic) employee handbook, MSA, invoice and annual report, and **thirteen real documents** — the four Labour Codes of 2019–2020, the Payment of Bonus, Gratuity, Maternity Benefit (with its 2017 amendment), POSH, DPDP, IT and CGST Acts, and the ministry's compliance handbook for employers — fetched from the publishers' own sites by `deploy/evals/fetch_real.py` (sha256 of every file in `real_sources.json`). The loader in the cell below is `deploy/shared/documind_corpus.py`, pasted verbatim: the same chunker as the ingest worker, the same chunk ids as the offline eval gate, so a citation here names the same page a production citation would. About 1,600 chunks and $0.05 of embeddings on a first run; nothing on a re-run.

Run 4.1 first and its Document AI chunks of the Gratuity Act are already there — the loader skips a document a tenant already holds, so the two lessons never write the same Act twice.


In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-firestore==2.30.0 pydantic
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.base_query import FieldFilter   # every tenant filter below
from pydantic import BaseModel, Field
from typing import List, Literal, Optional
import json, os, re, subprocess

client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings: regional only
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')   # Gemini 3.x generation: global only
db = firestore.Client(project=PROJECT_ID)

# --- Resources this lesson queries (chunks - THE collection): DB + vector index + THE corpus ---
CHUNKS_COLLECTION = 'chunks'   # one name, from 2.3 to production (deploy/services/rag-api/config.py)
TENANT_ID = 'acme'             # the teaching tenant; the kit filters by this field on every query
# 1) Firestore (default) database (idempotent)
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print('Creating Firestore (default) database...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=asia-south1', '--project', PROJECT_ID], check=False)

# 2) The SAME index the kit provisions (deploy/terraform/firestore_indexes.tf): tenant_id ASC +
#    embedding. Equality filter FIRST, vector field LAST - the order is load-bearing.
_idx = subprocess.run(
    "gcloud firestore indexes composite create --project=" + PROJECT_ID +
    " --collection-group=chunks --query-scope=COLLECTION"
    " --field-config=field-path=tenant_id,order=ascending"
    " --field-config=field-path=embedding,vector-config='{\"dimension\":\"768\",\"flat\":\"{}\"}'",
    shell=True, capture_output=True, text=True)
_out = (_idx.stdout + _idx.stderr).lower()
print('chunks tenant_id+embedding index:',
      'creating (~2-5 min to build)' if _idx.returncode == 0
      else ('already exists' if 'already exists' in _out else (_idx.stderr.strip()[:90])))

# 3) THE corpus. Not five typed sentences: DocuMind's documents, from deploy/evals/corpus/ in
#    the course repo (found beside this notebook, or cloned under /content the way 10.4 does).
#    ACME holds its synthetic handbook, MSA, invoice and report, and thirteen REAL documents -
#    the four Labour Codes, the Payment of Bonus, Gratuity, Maternity Benefit (with its 2017
#    amendment), POSH, DPDP, IT and CGST Acts, and the ministry's compliance handbook - fetched from
#    the publishers' own sites by deploy/evals/fetch_real.py, sha256 of every file in real_sources.json.
#    The loader below is deploy/shared/documind_corpus.py, pasted VERBATIM (tools/check_contract.py
#    holds it to that): the ingest worker's chunker, the offline gate's chunk ids, so a citation
#    here names the same page a production citation would. A document the tenant already holds
#    (4.1's Document AI chunks, an earlier run) is skipped, never written twice.
# --- kit: begin ---------------------------------------------------------------
import hashlib
KIT_REPO = "https://github.com/netsetos/agentic-ai-weekend-gcp-learners"   # the learner repo carries the kit under deploy/
KIT_BRANCH = "rag-production-hardening"   # the learner repo (public): the notebooks and the kit, deploy/, on its main branch
CHUNK_CHARS, CHUNK_OVERLAP = 2000, 200                     # services/ingest/main.py
EMBEDDING_MODEL, EMBEDDING_VERSION = "text-embedding-005", "1"   # stamped on every row; the worker reads the same pair from its environment (variables.tf)
RETENTION_DAYS = 30                                        # a retired row expires this long after it is superseded (the TTL policy in firestore_indexes.tf)
SCHEMA_VERSION = 2                                         # the row shape: doc_key/current (1); chunk_hash, locator, the embedding stamp, expire_at (2)
_SECTION = re.compile(r"^## +(.+?) *$", re.M)
_CODE = re.compile(r"^([A-Z][A-Z0-9]{0,7}(?:-[A-Z0-9]{1,6}){1,2})\b")   # NP-03, IT-SEC-04, MSA-04, GEN-014
_EFFECTIVE = re.compile(r"effective[ _-]?(?:from|date)?\s*[:=]\s*(\d{4}-\d{2}-\d{2})", re.I)   # services/ingest/contracts.py


def find_kit(start: str = ".") -> str:
    """The deploy/evals directory: beside the notebook, above it, or a clone under /content."""
    here = os.path.abspath(start)
    for _ in range(6):
        for cand in (os.path.join(here, "deploy", "evals"), os.path.join(here, "evals"), here):
            if os.path.isfile(os.path.join(cand, "manifest.json")) and os.path.isdir(os.path.join(cand, "corpus")):
                return cand
        here = os.path.dirname(here)
    clone = "/content/agentic-ai-weekend-gcp-learners"
    if not os.path.isdir(clone):
        subprocess.run(["git", "clone", "--depth", "1", "-b", KIT_BRANCH, KIT_REPO, clone], check=True)
    return os.path.join(clone, "deploy", "evals")


def load_documents(tenant: str, evals_dir: str, project_id: str) -> list:
    """Every document of one tenant that has text on disk: the synthetic .md files and the real
    Acts' pypdf mirrors. A scanned PDF with no mirror (posh_act_2013) is skipped - that one is
    lesson 4.1's, and only Document AI can read it. Each carries its VERSION - the sha256 of the
    OBJECT the lane ingests (the .md itself; for a real Act the PDF, never its mirror), which is
    the worker's doc_key for the same bytes, so a notebook and the lane name one version of one
    document (13 September 2026) - and the date it declares, if any."""
    docs = []
    for m in json.load(open(os.path.join(evals_dir, "manifest.json"), encoding="utf-8")):
        if m["tenant_id"] != tenant or not m.get("chars"):
            continue
        mirror = os.path.join(evals_dir, m["file"].rsplit(".", 1)[0] + ".md")
        if not os.path.isfile(mirror):
            continue
        raw = open(mirror, "rb").read()
        text = raw.decode("utf-8")
        dated = _EFFECTIVE.search(text[:3000])
        obj = os.path.join(evals_dir, m["file"])           # what the worker would hash: the PDF beside its mirror, or the .md
        version = (hashlib.sha256(open(obj, "rb").read()).hexdigest() if os.path.isfile(obj)
                   else m.get("sha256") or hashlib.sha256(raw).hexdigest())
        docs.append({"slug": m["slug"], "doc_type": m["doc_type"],
                     "source_uri": m["gcs_uri"].replace("${PROJECT_ID}", project_id),
                     "text": text, "sha256": version,
                     "mirror_sha256": hashlib.sha256(raw).hexdigest(),   # provenance: which mirror text was chunked
                     "effective_from": dated.group(1) if dated else None})
    return docs


def windows(text: str) -> list:
    """The worker's chunker: fixed windows with overlap, ending on a sentence when one is nearby."""
    text = text.strip()
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + CHUNK_CHARS)
        if end < len(text):
            cut = text.rfind(". ", start + CHUNK_CHARS // 2, end)
            if cut != -1:
                end = cut + 1
        piece = text[start:end].strip()
        if piece:
            out.append(piece)
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return out


def chunk_hash(text: str) -> str:
    """The chunk's identity across versions (services/ingest/contracts.py): the hash of its text with the
    whitespace collapsed. A re-wrapped paragraph is the same paragraph; a changed figure is a new chunk."""
    return hashlib.sha256(re.sub(r"\s+", " ", text).strip().encode("utf-8")).hexdigest()


def chunk_document(doc: dict, tenant: str, version: str = "") -> list:
    """Canonical chunk documents - the fields services/ingest/indexer.py writes - minus the embedding. Each carries
    its LOCATOR (the clause code or the section's ordinal; a page and its window for a PDF mirror; `preamble` for
    the text above a handbook's first heading) and its chunk_hash. The first version of a document keeps the classic
    id (acme:hr_policy_2026#NP-03); a re-issue carries its version in the id (acme:hr_policy_2026@1f3a9c2b#NP-03),
    so the retired rows stay beside the current ones and nothing is overwritten. An anchor matches either."""
    text = re.sub(r"\A\s*<!--.*?-->\s*", "", doc["text"], count=1, flags=re.S)   # a mirror's provenance header
    base = {"tenant_id": tenant, "source_uri": doc["source_uri"], "doc_type": doc["doc_type"], "kind": "text"}
    at = f"@{version}" if version else ""
    out = []
    heads = list(_SECTION.finditer(text))
    if heads:                                                  # a handbook: one chunk per section
        for k, piece in enumerate(windows(text[:heads[0].start()])):
            loc = "preamble" + (f"-{k}" if k else "")
            out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}{at}#{loc}", "text": piece, "page_start": 1,
                        "section": None, "locator": loc, "chunk_hash": chunk_hash(piece)})
        for n, h in enumerate(heads):
            body = text[h.end(): heads[n + 1].start() if n + 1 < len(heads) else len(text)].strip()
            title = h.group(1).strip()
            code = _CODE.match(title)
            key = code.group(1) if code else f"s{n + 1}"
            for k, piece in enumerate(windows(f"{title}\n{body}")):
                loc = key + (f"-{k}" if k else "")
                out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}{at}#{loc}", "text": piece, "page_start": 1,
                            "section": title, "locator": loc, "chunk_hash": chunk_hash(piece)})
    else:                                                      # a PDF mirror: pages split by \f
        for p, page in enumerate(text.split("\f"), 1):
            for k, piece in enumerate(windows(page)):
                loc = f"p{p}-{k}"
                out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}{at}#{loc}", "text": piece, "page_start": p,
                            "locator": loc, "chunk_hash": chunk_hash(piece)})
    return out


EMBED_BATCH, EMBED_TOKENS, CHARS_PER_TOKEN = 250, 15_000, 3


def embed_batches(texts: list) -> list:
    """Batches of at most EMBED_BATCH texts AND about EMBED_TOKENS tokens. text-embedding-005 takes
    250 texts per request and 20,000 tokens across them, and a request over either limit fails
    whole; a two-thousand-character chunk is ~500 tokens, so 250 of them are ~125,000. The first
    live corpus load (6 Sept 2026) failed every long Act exactly here - the same rule now lives in
    services/ingest/indexer.py."""
    out, cur, cur_tokens = [], [], 0
    for t in texts:
        tokens = max(1, len(t) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur); cur, cur_tokens = [], 0
        cur.append(t); cur_tokens += tokens
    if cur:
        out.append(cur)
    return out


def rows_of(db, tenant: str, source_uri: str, collection: str = "chunks") -> list:
    """Every row the tenant holds for one source - its version, its flags, its hash, its vector and its embedding
    stamp. One query, two equality filters, no composite index."""
    from google.cloud.firestore_v1.base_query import FieldFilter
    out = []
    for d in (db.collection(collection).where(filter=FieldFilter("tenant_id", "==", tenant))
                .where(filter=FieldFilter("source_uri", "==", source_uri)).stream()):
        x = d.to_dict() or {}
        out.append({"id": d.id, "ref": d.reference, "doc_key": x.get("doc_key"), "current": x.get("current"),
                    "staged": x.get("staged"), "chunk_hash": x.get("chunk_hash"), "embedding": x.get("embedding"),
                    "embedding_model": x.get("embedding_model"), "embedding_version": x.get("embedding_version")})
    return out


def swap_versions(db, rows: list, new_doc_key: str, retention_days: int = RETENTION_DAYS, effective_to: str = None) -> dict:
    """Visibility is a swap (services/ingest/idempotency.py, the same rule). Every row of new_doc_key becomes current -
    a staged re-issue, or the retired rows of a version uploaded again (the undo) - and then every OTHER current row
    of the source is retired: a flag, superseded_by, and expire_at = now + retention_days for the TTL policy. Never a
    delete. Batches of 400; a reader between two batches sees the new version only (the newest-per-source guard)."""
    import datetime
    from google.cloud import firestore
    expire = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=retention_days)
    activated, retired, n = 0, 0, 0
    batch = db.batch()
    for r in rows:
        if r["doc_key"] == new_doc_key and r["current"] is not True:
            fields = {"current": True, "staged": firestore.DELETE_FIELD, "expire_at": firestore.DELETE_FIELD,
                      "superseded_by": firestore.DELETE_FIELD, "superseded_at": firestore.DELETE_FIELD,
                      "effective_to": firestore.DELETE_FIELD}
            if not r.get("staged"):
                fields["reactivated_at"] = firestore.SERVER_TIMESTAMP     # the undo: a retired row, current again
            batch.update(r["ref"], fields)
            activated += 1; n += 1
        elif r["doc_key"] != new_doc_key and r["current"] is not False:
            fields = {"current": False, "superseded_by": new_doc_key, "superseded_at": firestore.SERVER_TIMESTAMP,
                      "expire_at": expire}
            if effective_to:
                fields["effective_to"] = effective_to
            batch.update(r["ref"], fields)
            retired += 1; n += 1
        if n and n % 400 == 0:                                   # a Firestore batch holds 500 writes
            batch.commit(); batch = db.batch()
    if n % 400:
        batch.commit()
    return {"activated": activated, "retired": retired}


def record_version(db, tenant: str, source_uri: str, doc_key: str, sha256: str, chunks: int, reused: int,
                   embedded: int, retired: int, effective_from: str = None) -> str:
    """The ledger row - sources/{tenant~name}: which version is current and what the reindex cost - and the tenant's
    corpus fingerprint, ledger/{tenant}: the hash of its current doc_keys, the key rag-api's cache follows. What the
    worker writes on every ingest (services/ingest/idempotency.py: record_source, refresh_fingerprint), from a notebook."""
    from google.cloud import firestore
    from google.cloud.firestore_v1.base_query import FieldFilter
    name = source_uri.split("/", 3)[-1]                        # gs://bucket/<tenant>/<file> -> <tenant>/<file>
    db.collection("sources").document(name.replace("/", "~")).set(
        {"tenant_id": tenant, "name": name, "gcs_uri": source_uri, "doc_key": doc_key, "generation": "notebook",
         "sha256": sha256, "chunks": chunks, "reused": reused, "embedded": embedded, "retired": retired,
         "effective_from": effective_from, "status": "indexed", "embedding_model": EMBEDDING_MODEL,
         "embedding_version": EMBEDDING_VERSION, "indexed_at": firestore.SERVER_TIMESTAMP}, merge=True)
    keys = sorted((s.to_dict() or {}).get("doc_key") or "" for s in
                  db.collection("sources").where(filter=FieldFilter("tenant_id", "==", tenant))
                    .where(filter=FieldFilter("status", "==", "indexed")).stream())
    fp = hashlib.sha256("\n".join(keys).encode("utf-8")).hexdigest()[:16]
    db.collection("ledger").document(tenant).set(
        {"tenant_id": tenant, "fingerprint": fp, "versions": len(keys), "last_event": "notebook_seed",
         "updated_at": firestore.SERVER_TIMESTAMP}, merge=True)
    return fp


def seed(db, embed, tenant: str, project_id: str, evals_dir: str = None, collection: str = "chunks",
         retention_days: int = RETENTION_DAYS) -> dict:
    """Write one tenant's corpus into Firestore, version-aware and idempotent (12 September 2026).

    A document's version is the hash of its bytes - the worker's doc_key, tenant_sha256, for the same bytes.
    The tenant holds this version, current: nothing to do. Holds it retired (a later version replaced it):
    the UNDO - its rows come back current, the later version is retired, nothing is embedded. Holds another
    version: the RE-ISSUE - the new chunks are written staged, every unchanged chunk's vector reused by
    chunk_hash and only the changed ones sent to `embed`, then one swap makes them current and retires the old
    rows with expire_at. Holds rows with no version at all (4.1's Document AI chunks on a lane older than the
    ledger): left alone, never written twice. Holds nothing: the first version, written current.
    `embed(texts) -> vectors` is the notebook's batched text-embedding-005 call. Returns {slug: chunks written}
    and prints one line per version event."""
    from google.cloud import firestore
    from google.cloud.firestore_v1.vector import Vector
    evals_dir = evals_dir or find_kit()
    counts = {}
    for doc in load_documents(tenant, evals_dir, project_id):
        key = f"{tenant}_{doc['sha256']}"
        held = rows_of(db, tenant, doc["source_uri"], collection)
        live = [r for r in held if r["current"] is not False and not r.get("staged")]
        counts[doc["slug"]] = 0
        if any(r["doc_key"] == key for r in live) or (held and not any(r["doc_key"] for r in held)):
            continue                                             # this version is current, or the rows predate versions
        if any(r["doc_key"] == key and r["current"] is False for r in held):
            n = swap_versions(db, held, key, retention_days, doc.get("effective_from"))
            record_version(db, tenant, doc["source_uri"], key, doc["sha256"], n["activated"], n["activated"], 0,
                           n["retired"], doc.get("effective_from"))
            print(f"  {doc['slug']}: reactivated {n['activated']} chunks, retired {n['retired']}, embedded 0 (the undo)")
            continue
        chunks = chunk_document(doc, tenant, doc["sha256"][:8] if live else "")
        if not chunks:
            continue
        by_hash = {r["chunk_hash"]: r["embedding"] for r in live
                   if r["chunk_hash"] and r["embedding"] is not None
                   and r["embedding_model"] == EMBEDDING_MODEL and str(r["embedding_version"]) == EMBEDDING_VERSION}
        vectors = [by_hash.get(c["chunk_hash"]) for c in chunks]           # the carry-over: reused by hash
        misses = [c["text"] for c, v in zip(chunks, vectors) if v is None]
        fresh = []
        for texts in embed_batches(misses):                                  # 250 texts AND 20,000 tokens per request
            fresh += embed(texts)
        it = iter(fresh)
        vectors = [list(v) if v is not None else next(it) for v in vectors]
        batch, n = db.batch(), 0
        for c, v in zip(chunks, vectors):
            row = {**c, "doc_key": key, "current": not live, "embedding": Vector(v),
                   "embedding_model": EMBEDDING_MODEL, "embedding_version": EMBEDDING_VERSION,
                   "schema_version": SCHEMA_VERSION, "indexed_at": firestore.SERVER_TIMESTAMP,
                   "processed_at": firestore.SERVER_TIMESTAMP}
            if doc.get("effective_from"):
                row["effective_from"] = doc["effective_from"]
            if live:
                row["staged"] = True                                 # a re-issue lands invisible; the swap makes it current
            batch.set(db.collection(collection).document(c["chunk_id"]), row)
            n += 1
            if n % 400 == 0:                                         # a Firestore batch holds 500 writes
                batch.commit()
                batch = db.batch()
        batch.commit()
        retired = 0
        if live:
            retired = swap_versions(db, rows_of(db, tenant, doc["source_uri"], collection), key, retention_days,
                                    doc.get("effective_from"))["retired"]
            print(f"  {doc['slug']}: revision {doc['sha256'][:8]}: {len(chunks) - len(misses)} chunks reused by hash, "
                  f"{len(misses)} embedded, {retired} retired")
        record_version(db, tenant, doc["source_uri"], key, doc["sha256"], len(chunks), len(chunks) - len(misses),
                       len(misses), retired, doc.get("effective_from"))
        counts[doc["slug"]] = len(chunks)
    return counts


# Seed THE corpus (a re-run skips every document the tenant already holds - nothing is embedded twice).
EVALS_DIR = find_kit()

def embed_documents(texts):
    """Batched, RETRIEVAL_DOCUMENT: seed() calls this once per batch of at most 250 texts (2.2's rule)."""
    return [e.values for e in client.models.embed_content(
        model=EMBEDDING_MODEL, contents=texts,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)).embeddings]

written = seed(db, embed_documents, TENANT_ID, PROJECT_ID, EVALS_DIR, CHUNKS_COLLECTION)
print(f'corpus: {sum(written.values())} chunks written for tenant {TENANT_ID!r}, '
      f'{sum(1 for n in written.values() if n == 0)} documents already held '
      '(the scanned POSH Act has no text layer - 4.1 reads it with Document AI)')


In [ ]:
import json
from pydantic import ValidationError

def draft_of(r, schema, quote_limit=200):
    """The model's structured answer, or a clear error - never a silent None.

    The SDK sets r.parsed to None on ANY validation failure. The first live run of the lane
    (7 Sept 2026) met the one that matters: a quote longer than the contract's 200 characters -
    a statute provision is one sentence - which threw fourteen right answers away and, worse,
    had been scored as the model refusing. So: read the JSON, trim the quote to the contract,
    validate again; anything else is an error you can read, not a refusal."""
    if r.parsed is not None:
        return r.parsed if isinstance(r.parsed, schema) else schema.model_validate(r.parsed)
    cand = (r.candidates or [None])[0]
    reason = getattr(getattr(cand, "finish_reason", None), "name", "NO_CANDIDATES")
    try:
        obj = json.loads(r.text or "")
    except ValueError:
        raise RuntimeError(f"no JSON to parse (finish_reason={reason}) - raise max_output_tokens if it is MAX_TOKENS")
    if isinstance(obj, dict):
        for c in obj.get("citations") or []:
            if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > quote_limit:
                c["quote"] = c["quote"][: quote_limit - 3].rstrip() + "..."
    try:
        return schema.model_validate(obj)
    except ValidationError as e:
        err = e.errors()[0]
        raise RuntimeError(f"the draft failed the contract at {'.'.join(str(x) for x in err['loc'])}: {err['msg']}") from None


### Wait for the vector index to build (run after Setup)
This blocks until the Firestore vector index reports `READY` — about 2-5 minutes on the **first** run only (it returns instantly afterwards). The retrieval cells below raise `FAILED_PRECONDITION` until this prints READY. Safe to re-run any time.

In [ ]:
# 4) Block until the chunks vector index is READY (find_nearest raises
#    FAILED_PRECONDITION until then). Mirrors lesson 2.3's Setup safeguard so a
#    top-to-bottom / Run-all does not hit the retrieval cells while the index builds.
import subprocess, time
print('Waiting for chunks vector index to reach READY (~2-5 min; retrieval fails until then)...')
for _ in range(40):  # up to ~10 min
    _rows = subprocess.run(
        ['gcloud', 'firestore', 'indexes', 'composite', 'list',
         '--project', PROJECT_ID, '--format=value(name,state)'],
        capture_output=True, text=True).stdout.lower()
    _rag = [ln for ln in _rows.splitlines() if '/chunks' in ln or ' chunks' in ln or ln.startswith('chunks')]
    if _rag and all('creating' not in ln for ln in _rag) and any('ready' in ln for ln in _rag):
        print('chunks vector index READY - retrieval cells (2, 6, 8) will work now.')
        break
    time.sleep(15)
else:
    print('Index still building after ~10 min. Wait a bit, then re-run THIS cell.')

## Cell 1: Stage 1 — Embed the Query


In [ ]:
def embed_query(query):
    result = client.models.embed_content(
        model='text-embedding-005', contents=query,
        config=types.EmbedContentConfig(
            task_type='RETRIEVAL_QUERY',
            output_dimensionality=768))
    return result.embeddings[0].values

qv = embed_query('What is the overtime rate under the Code on Wages?')
print(f'Query vector: {len(qv)} dims, first 5: {qv[:5]}')


## Cell 2: Stage 2 — Retrieve from Firestore

Two rules the lane added on 12 September 2026, and this cell's `if not chunks` is the first of them in miniature. An empty pool is answered **without a model call**: `/v1/query` returns the contract's refusal (`answerable=false`, no citations, confidence `low`) at zero tokens and zero cost, because a model shown nothing can only invent or decline at full price. And a request's `filters` may name `doc_type` and `kind` only; any other key is a **400** that lists the allowed keys, never an empty pool that reads like an honest "nothing found" - `tenant_id` comes from the roster and `current` from the ledger, neither from the caller.


In [ ]:
def retrieve_chunks(query_vector, collection=CHUNKS_COLLECTION, top_k=5,
                     distance_threshold=0.5, tenant_id=TENANT_ID):
    """Retrieve top-K relevant chunks from Firestore vector index.

    Filtered by tenant FIRST - there is no unfiltered query on a multi-tenant store, and
    the composite index (tenant_id ASC + embedding) exists precisely for this shape."""
    results = (
        db.collection(collection)
        .where(filter=FieldFilter("tenant_id", "==", tenant_id))
        .find_nearest(
            vector_field="embedding",
            query_vector=Vector(query_vector),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k,
            distance_threshold=distance_threshold,  # Quality gate: 0.5, not 0.3 - a right statute window sits
                                                    # farther from a question than a short synthetic clause does;
                                                    # the printed distances below are the numbers to tune this on
            distance_result_field="vector_distance",  # 👉 writes distance into each doc
        )
        .get()
    )

    chunks = []
    for doc in results:
        data = doc.to_dict()
        if data.get("current") is False:            # the ledger (12.5): a retired version is never a source; the lane
            continue                                # pre-filters current==true with its second vector index
        distance = data.get("vector_distance")  # COSINE distance (0=identical)
        similarity = 1 - distance if distance is not None else 0
        chunks.append({
            "id": doc.id,                                   # what resolve() reads
            "chunk_id": doc.id,
            "text": data.get("text", ""),
            "source_uri": data.get("source_uri", "unknown"),
            "page_start": data.get("page_start"),
            "score": similarity,
            "similarity": similarity,
            "distance": distance,
        })
    return chunks

chunks = retrieve_chunks(qv)
print(f'Retrieved {len(chunks)} chunks')
for c in chunks:
    print(f'  [sim {c["similarity"]:.2f} | distance {c["distance"]:.3f}] {c["source_uri"].rsplit("/", 1)[-1]} | {c["text"][:60]}...')
print('distance is COSINE distance (0 = identical); anything at or above the threshold was never returned')


## Cell 3: Stage 3 — Augment Prompt


In [ ]:
def build_rag_prompt(query, chunks):
    if not chunks:
        return f'Question: {query}\n\nNo relevant context found.'
    ctx = '\n\n'.join(
        f'[Source {i+1}] ({c["source_uri"].rsplit("/", 1)[-1]}, p.{c["page_start"]})\n{c["text"]}'
        for i, c in enumerate(chunks))
    return f'Context:\n{ctx}\n\nQuestion: {query}'

prompt = build_rag_prompt('What is the overtime rate under the Code on Wages?', chunks)
print(prompt[:400])


## Cell 4: Stage 4 — Generate with Citations


In [ ]:
# THE contract - verbatim from deploy/shared/documind_schemas.py, the same text 3.2 teaches
# and the rag-api service imports. One definition; this cell does not retype it.
class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool

def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)

RAG_SYSTEM = '''You are DocuMind. Answer ONLY from context.
Cite every claim with [Source N] and quote the words you relied on. Rate confidence.
A quote is the clause that answers - at most twenty-five words, never a whole section.'''
# The last rule is the lane's own (generator.py, rule 5): a quote over the contract's 200 characters fails
# validation and the SDK hands back None for the whole draft - on statutes, constantly (4.8, F21).

def generate_rag(prompt, chunks):
    """Ask for a ModelDraft (cites by [Source N]); resolve() it against the chunks the model
    saw. The model never sees a chunk id, a page or a score - those come from `chunks`."""
    r = gen_client.models.generate_content(
        model='gemini-3.6-flash', contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=RAG_SYSTEM,
            response_mime_type='application/json',
            response_schema=ModelDraft,
            # temperature / top_p / top_k removed: gemini-3.6-flash ignores them (model page, GA 2026-07-21)
            thinking_config=types.ThinkingConfig(thinking_level="LOW")))
    return resolve(draft_of(r, ModelDraft), chunks), r.usage_metadata

result, usage = generate_rag(prompt, chunks)
print(f'Answer: {result.answer[:200]}...')
print(f'Confidence: {result.confidence} | Answerable: {result.answerable}')
print(f'Citations: {len(result.citations)}')
for c in result.citations:
    print(f'  {c.chunk_id:8} {c.source_uri.rsplit("/", 1)[-1]:22} score={c.score:.2f}  "{c.quote[:60]}"')


## Cell 5: Stage 5 — Cost Tracking


In [ ]:
def calc_cost(usage, n_chunks=5):
    # 1.50 / 7.50 per 1M tokens = gemini-3.6-flash Vertex AI standard rate from 2027-01-01
    # (intro $0.75/$3.75 through 2026-12-31 - see lesson 1.3); Rs at USD_INR = 85
    inp = usage.prompt_token_count
    out = usage.candidates_token_count
    think = usage.thoughts_token_count or 0
    cost = inp/1e6*1.50 + (out+think)/1e6*7.50 + 0.00001
    print(f'Input: {inp} tokens (${inp/1e6*1.50:.6f})')
    print(f'Output: {out} tokens (${out/1e6*7.50:.6f})')
    print(f'Thinking: {think} tokens')
    print(f'Total: ${cost:.6f} (Rs {cost*85:.4f})')
    return cost

calc_cost(usage)


## Cell 6: Complete Pipeline — ask_documind()


In [ ]:
def ask_documind(query, top_k=5):
    qv = embed_query(query)
    chunks = retrieve_chunks(qv, top_k=top_k)
    prompt = build_rag_prompt(query, chunks)
    if not chunks:   # the lane's rule too (12.2, 12 September 2026): an empty pool is refused before any model call
        return {'answer':'No relevant context.','confidence':'low','citations':[],'answerable':False,'cost_usd':0}
    result, usage = generate_rag(prompt, chunks)
    cost = usage.prompt_token_count/1e6*1.50 + (usage.candidates_token_count + (usage.thoughts_token_count or 0))/1e6*7.50
    return {
        'answer': result.answer,
        'confidence': result.confidence,
        'citations': [c.model_dump() for c in result.citations],
        'answerable': result.answerable,
        'cost_usd': cost}

# A real question with a real answer on a real page: section 14 of the Code on Wages, 2019.
r = ask_documind('What is the overtime rate under the Code on Wages?')
print(f'Answer: {r["answer"][:150]}...')
print(f'Confidence: {r["confidence"]}')
print(f'Citations: {len(r["citations"])}')
for c in r['citations']:
    print(f'  {c["source_uri"].rsplit("/", 1)[-1]} p.{c["page"]}: "{c["quote"][:90]}"')
print(f'Cost: ${r["cost_usd"]:.6f}')


## Cell 7: Citation Verification


In [ ]:
def verify_citations(ans, chunks):
    """resolve() already dropped any [Source N] outside the context, so 'invalid source id' can
    never reach this function. What CAN still fail: a quote the cited chunk does not contain
    (the model paraphrased or invented it), and claims with no citation at all."""
    by_id = {c['chunk_id']: c for c in chunks}
    issues = []
    for c in ans.citations:
        src = by_id.get(c.chunk_id)
        if src is None:
            issues.append(f'citation to a chunk the model never saw: {c.chunk_id}')
        elif ' '.join(c.quote.split()).lower() not in ' '.join(src['text'].split()).lower():
            issues.append(f'quote not found in {c.chunk_id}: "{c.quote[:50]}"')
    sentences = ans.answer.split('. ')
    uncited = sum(1 for s in sentences if '[Source' not in s)
    if uncited > 1:
        issues.append(f'{uncited} uncited sentences')
    return {'valid': len(issues)==0, 'issues': issues}

v = verify_citations(result, chunks)
print(f'Valid: {v["valid"]}')
for issue in v['issues']:
    print(f'  Issue: {issue}')


## Cell 8: RAGEngine Production Class


In [ ]:
class RAGEngine:
    def __init__(self, project, location='us-central1'):
        self.client = genai.Client(enterprise=True, project=project, location=location)      # embeddings: regional
        self.gen_client = genai.Client(enterprise=True, project=project, location='global')  # generation: global
        self.db = firestore.Client(project=project)
        self.total_cost = 0.0
        self.query_count = 0

    def query(self, question, collection=CHUNKS_COLLECTION, top_k=5, tenant_id=TENANT_ID):
        qv = self.client.models.embed_content(
            model='text-embedding-005', contents=question,
            config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY',
                                            output_dimensionality=768)
        ).embeddings[0].values
        docs = self.db.collection(collection).where(filter=FieldFilter('tenant_id', '==', tenant_id)).find_nearest(
            vector_field='embedding', query_vector=Vector(qv),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k, distance_threshold=0.5).get()
        chunks = [{'id':d.id,'text':d.to_dict().get('text',''),
                   'source_uri':d.to_dict().get('source_uri',''),
                   'page_start':d.to_dict().get('page_start'), 'score':0.0} for d in docs]
        if not chunks:
            return {'answer':'No relevant context.','confidence':'low'}
        ctx = '\n\n'.join(f'[Source {i+1}]\n{c["text"]}' for i,c in enumerate(chunks))
        r = self.gen_client.models.generate_content(
            model='gemini-3.6-flash', contents=f'Context:\n{ctx}\n\nQuestion: {question}',
            config=types.GenerateContentConfig(
                system_instruction=RAG_SYSTEM,
                response_mime_type='application/json', response_schema=ModelDraft,
                thinking_config=types.ThinkingConfig(thinking_level="LOW")))
        cost = r.usage_metadata.prompt_token_count/1e6*1.50 + (r.usage_metadata.candidates_token_count + (r.usage_metadata.thoughts_token_count or 0))/1e6*7.50
        self.total_cost += cost
        self.query_count += 1
        return {'result':resolve(draft_of(r, ModelDraft), chunks), 'cost':cost, 'chunks':len(chunks)}

    def report(self):
        avg = self.total_cost/self.query_count if self.query_count else 0
        print(f'Queries: {self.query_count} | Total: ${self.total_cost:.4f} | Avg: ${avg:.6f}/q')

print('RAGEngine ready')


## ✅ Lesson 4.2 Complete!

- ✅ 5-stage RAG: embed → retrieve → augment → generate → return — over DocuMind's real documents, with a page number on every citation
- ✅ RETRIEVAL_QUERY vs RETRIEVAL_DOCUMENT task types
- ✅ Firestore find_nearest() with distance_threshold quality gate
- ✅ Numbered [Source N] context for citation tracking
- ✅ The shared contract: ModelDraft (what the model is asked for) → resolve() → RAGAnswer with full Citations — the same three classes 3.2 teaches and rag-api imports
- ✅ Per-query cost tracking in USD and INR
- ✅ Citation verification and hallucination detection
- ✅ RAGEngine production module
- ✅ One corpus: `deploy/evals/corpus/` through `deploy/shared/documind_corpus.py`, the same chunks and ids the ingest worker and the eval gate use

4.5 keeps this retrieve → pack → generate shape and adds a BM25 retriever fused with RRF, Rank API reranking, a TokenBudget for packing and an explicit per-tenant cache; 4.7 puts a faithfulness gate in front of it.

**Next: Lesson 4.3 — Vertex AI RAG Engine (the managed version of this pipeline); 4.5 adds hybrid retrieval, reranking and budgeted packing on top of these chunks**
